In [1]:
# CSEC720 final project
# Fall 2026
# Cayden Wright, Eric Antonecchia

import hashlib
import json
import random
from io import BytesIO
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_CLASSES = 43
SEED = 720
ROOT = Path.cwd().resolve()
MODEL_PATH = ROOT / 'gtsrb_cnn.pth'
ADVERSARIAL_CACHE_DIR = ROOT / 'adversarial_cache'
MEAN = (0.3403, 0.3121, 0.3214)
STD = (0.2724, 0.2608, 0.2669)
EPSILONS = (0.0, 0.01, 0.03, 0.06)
PREVIEW_EPSILON = 0.03
PREPROCESSING = {'dataset': 'GTSRB', 'resize': [32, 32],
                 'mean': MEAN, 'std': STD, 'attack_space': 'raw_pixels_0_1'}
# Keep images in pixel space for attacks and later image transformations.
preprocess = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])


class TrafficSignCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 512), nn.ReLU(),
            nn.Dropout(0.5), nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class PixelSpaceModel(nn.Module):
    """Accept raw pixels while preserving the original CNN's normalization."""
    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier
        self.register_buffer('mean', torch.tensor(MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(STD).view(1, 3, 1, 1))

    def forward(self, x):
        return self.classifier((x - self.mean) / self.std)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)


def make_loaders():
    train = datasets.GTSRB(root=str(ROOT / 'data'), split='train', download=True, transform=preprocess)
    test = datasets.GTSRB(root=str(ROOT / 'data'), split='test', download=True, transform=preprocess)
    train_size = int(0.8 * len(train))
    train, val = random_split(train, [train_size, len(train) - train_size],
                              generator=torch.Generator().manual_seed(SEED))
    def loader(dataset, shuffle):
        return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2,
                          worker_init_fn=seed_worker,
                          generator=torch.Generator().manual_seed(SEED))
    return loader(train, True), loader(val, False), loader(test, False)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total = 0.0, 0
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            logits = model(inputs)
            total_loss += criterion(logits, labels).item() * len(labels)
            predictions = logits.argmax(1)
            pairs = (labels * NUM_CLASSES + predictions).cpu()
            confusion += torch.bincount(pairs, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)
            total += len(labels)
    if not total:
        raise ValueError('Cannot evaluate an empty dataset')
    tp = confusion.diag().float()
    denominator = confusion.sum(0) + confusion.sum(1)
    f1 = torch.where(denominator > 0, 2 * tp / denominator.clamp_min(1), 0.0)
    return total_loss / total, tp.sum().item() / total, f1.mean().item()


def validate_attack_inputs(inputs, epsilon):
    if epsilon < 0 or not np.isfinite(epsilon):
        raise ValueError('epsilon must be finite and nonnegative')
    if not torch.isfinite(inputs).all() or inputs.min() < 0 or inputs.max() > 1:
        raise ValueError('Attacks require finite raw pixels in [0, 1]')


def fgsm_attack(model, inputs, labels, criterion, epsilon=0.03):
    model.eval()
    inputs = inputs.detach().clone().to(DEVICE)
    validate_attack_inputs(inputs, epsilon)
    if epsilon == 0:
        return inputs
    inputs.requires_grad_(True)
    loss = criterion(model(inputs), labels.to(DEVICE))
    gradient, = torch.autograd.grad(loss, inputs)
    return (inputs + epsilon * gradient.sign()).clamp(0, 1).detach()


def pgd_attack(model, inputs, labels, criterion, epsilon=0.03, alpha=0.005,
               iters=20, restarts=2, seed=SEED):
    model.eval()
    inputs = inputs.detach().clone().to(DEVICE)
    labels = labels.to(DEVICE)
    validate_attack_inputs(inputs, epsilon)
    if iters < 1 or restarts < 1 or alpha <= 0 or not np.isfinite(alpha):
        raise ValueError('PGD needs positive steps, restarts and finite step size')
    if epsilon == 0:
        return inputs
    generator = torch.Generator(device=inputs.device).manual_seed(seed)
    with torch.no_grad():
        best = inputs.clone()
        best_loss = nn.functional.cross_entropy(model(best), labels, reduction='none')
        best_success = model(best).argmax(1).ne(labels)

    # Retain successful examples and the strongest loss per image across every iterate.
    for _ in range(restarts):
        noise = torch.empty_like(inputs).uniform_(-epsilon, epsilon, generator=generator)
        adv = (inputs + noise).clamp(0, 1)
        for step in range(iters + 1):
            adv = adv.detach().requires_grad_(True)
            logits = model(adv)
            losses = nn.functional.cross_entropy(logits, labels, reduction='none')
            with torch.no_grad():
                success = logits.argmax(1).ne(labels)
                improve = (success & ~best_success) | ((success == best_success) & (losses > best_loss))
                best[improve] = adv.detach()[improve]
                best_loss[improve] = losses.detach()[improve]
                best_success[improve] = success[improve]
            if step == iters:
                break
            gradient, = torch.autograd.grad(losses.mean(), adv)
            adv = adv.detach() + alpha * gradient.sign()
            adv = torch.maximum(torch.minimum(adv, inputs + epsilon), inputs - epsilon).clamp(0, 1)
    return best.detach()


def model_fingerprint(model):
    digest = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        digest.update(name.encode())
        value = value.detach().cpu().contiguous()
        digest.update(str((value.dtype, tuple(value.shape))).encode())
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def load_or_generate_adversarial_dataset(model, loader, criterion, attack_name, attack_fn, attack_kwargs):
    """Cache exact ordered inputs, model identity, settings and clean predictions."""
    model.eval()
    clean_images, labels = [], []
    for batch, target in loader:
        clean_images.append(batch.cpu())
        labels.append(target.cpu())
    clean_images, labels = torch.cat(clean_images), torch.cat(labels)
    digest = hashlib.sha256(clean_images.numpy().tobytes())
    digest.update(labels.numpy().tobytes())
    metadata = {'version': 2, 'model': model_fingerprint(model), 'inputs': digest.hexdigest(),
                'preprocessing': PREPROCESSING, 'attack': attack_name, 'parameters': attack_kwargs,
                'batch_size': BATCH_SIZE, 'seed': SEED, 'torch': str(torch.__version__),
                'device': str(DEVICE), 'loss': 'cross_entropy'}
    # JSON roundtrip gives metadata the same types on save and reload.
    metadata = json.loads(json.dumps(metadata, sort_keys=True))
    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()
    ADVERSARIAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = ADVERSARIAL_CACHE_DIR / f'v2_{attack_name}_{key}.pt'
    if cache_path.exists():
        cached = torch.load(cache_path, map_location='cpu', weights_only=True)
        if cached['metadata'] != metadata:
            raise ValueError('Adversarial cache metadata mismatch')
        print(f'Loading cached {attack_name.upper()} images: {cache_path.name}')
    else:
        images, clean_predictions = [], []
        paired = DataLoader(TensorDataset(clean_images, labels), batch_size=BATCH_SIZE, shuffle=False)
        for batch_index, (inputs, targets) in enumerate(paired):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            with torch.no_grad():
                clean_predictions.append(model(inputs).argmax(1).cpu())
            kwargs = dict(attack_kwargs)
            if attack_fn is pgd_attack:
                kwargs['seed'] = SEED + batch_index
            with torch.enable_grad():
                adversarial = attack_fn(model, inputs, targets, criterion, **kwargs)
            if (adversarial - inputs).abs().max().item() > attack_kwargs['epsilon'] + 1e-6:
                raise ValueError('Attack exceeded its pixel-space perturbation budget')
            images.append(adversarial.cpu())
        cached = {'images': torch.cat(images), 'labels': labels,
                  'clean_predictions': torch.cat(clean_predictions),
                  'sample_ids': torch.arange(len(labels)), 'metadata': metadata}
        torch.save(cached, cache_path)
        print(f'Saved {attack_name.upper()} images: {cache_path.name}')
    return cached


def jpeg_compression(inputs, quality):
    decoded = []
    for image in inputs:
        pixels = image.mul(255).round().to(torch.uint8)
        with BytesIO() as buffer:
            TF.to_pil_image(pixels).save(buffer, format='JPEG', quality=quality)
            buffer.seek(0)
            with Image.open(buffer) as compressed:
                decoded.append(TF.to_tensor(compressed.convert('RGB')))
    return torch.stack(decoded)


def resize_round_trip(inputs, size):
    smaller = TF.resize(inputs, list(size), antialias=True)
    return TF.resize(smaller, list(inputs.shape[-2:]), antialias=True)


def reduce_bit_depth(inputs, bits):
    levels = 2 ** bits - 1
    return inputs.mul(levels).round().div(levels)


TRANSFORMATIONS = (
    ('jpeg_compression', jpeg_compression, {'quality': 75}),
    ('gaussian_blur', TF.gaussian_blur, {'kernel_size': [3, 3], 'sigma': [1.0, 1.0]}),
    ('resizing', resize_round_trip, {'size': [24, 24]}),
    ('bit_depth_reduction', reduce_bit_depth, {'bits': 4}),
)


@torch.no_grad()
def score_perturbed_images(model, cached, transform=None, parameters=None):
    model.eval()
    correct_clean = cached['clean_predictions'].eq(cached['labels'])
    batches = DataLoader(
        TensorDataset(cached['images'], cached['labels'], correct_clean), batch_size=BATCH_SIZE
    )
    correct, successes = 0, 0
    for inputs, labels, clean_correct in batches:
        if transform is not None:
            inputs = transform(inputs, **parameters)
        predictions = model(inputs.to(DEVICE)).argmax(1).cpu()
        matches = predictions.eq(labels)
        correct += matches.sum().item()
        successes += (~matches & clean_correct).sum().item()
    count = correct_clean.sum().item()
    return correct / len(cached['labels']), successes / count if count else None


def evaluate_adversarial(model, loader, criterion, attack_fn, attack_kwargs):
    name = 'fgsm' if attack_fn is fgsm_attack else 'pgd'
    cached = load_or_generate_adversarial_dataset(model, loader, criterion, name, attack_fn, attack_kwargs)
    return score_perturbed_images(model, cached)


def show_image_comparison(clean_image, label, perturbed_samples, epsilon):
    """Display one clean image and each independently transformed variant."""
    column_titles = ['Perturbed', 'JPEG', 'Gaussian blur', 'Resize', 'Bit-depth reduction']
    tile_size, column_width, row_height = 112, 136, 142
    left_margin, top_margin = 70, 36
    canvas = Image.new('RGB', (left_margin + column_width * len(column_titles),
                               top_margin + row_height * 3), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((12, 12), f'GTSRB test sample 0 | Class {label} | Epsilon {epsilon:g}', fill='black')

    for column, title in enumerate(column_titles):
        draw.text((left_margin + column * column_width, top_margin + row_height - 4),
                  title, fill='black')

    rows = [('Clean', [clean_image])]
    for attack_name, sample in perturbed_samples.items():
        variants = [transform(sample, **parameters)[0]
                    for _, transform, parameters in TRANSFORMATIONS]
        rows.append((attack_name, [sample[0], *variants]))

    for row_index, (name, images) in enumerate(rows):
        y = top_margin + row_index * row_height + (0 if row_index == 0 else 14)
        draw.text((12, y + tile_size // 2), name, fill='black')
        for column, image in enumerate(images):
            pixels = image.clamp(0, 1).mul(255).round().to(torch.uint8)
            tile = TF.to_pil_image(pixels).resize((tile_size, tile_size), Image.Resampling.NEAREST)
            canvas.paste(tile, (left_margin + column * column_width, y))
    display(canvas)


def main():
    seed_everything(SEED)
    train_loader, val_loader, test_loader = make_loaders()
    cnn = TrafficSignCNN().to(DEVICE)
    model = PixelSpaceModel(cnn).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    if MODEL_PATH.exists():
        saved = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
        # Older checkpoints contain only the CNN state dictionary.
        cnn.load_state_dict(saved['model_state_dict'] if 'model_state_dict' in saved else saved)
        print(f'Loaded checkpoint: {MODEL_PATH}')
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        best_accuracy, best_loss = -1.0, float('inf')
        for epoch in range(1, EPOCHS + 1):
            model.train()
            running_loss = 0.0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * len(labels)
            val_loss, val_accuracy, _ = evaluate(model, val_loader, criterion)
            print(f'Epoch {epoch}/{EPOCHS}: train loss={running_loss / len(train_loader.dataset):.4f}, '
                  f'validation loss={val_loss:.4f}, validation accuracy={val_accuracy:.4f}')
            if val_accuracy > best_accuracy or (val_accuracy == best_accuracy and val_loss < best_loss):
                best_accuracy, best_loss = val_accuracy, val_loss
                torch.save({'model_state_dict': cnn.state_dict(), 'epoch': epoch,
                            'validation_accuracy': val_accuracy, 'validation_loss': val_loss,
                            'seed': SEED, 'preprocessing': PREPROCESSING}, MODEL_PATH)
        saved = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
        cnn.load_state_dict(saved['model_state_dict'])
        print(f'Restored best validation checkpoint from epoch {saved["epoch"]}')

    loss, accuracy, macro_f1 = evaluate(model, test_loader, criterion)
    print(f'Clean test loss={loss:.4f}, accuracy={accuracy:.4f}, macro F1={macro_f1:.4f}')
    rows, transformation_rows = [], []
    perturbed_samples = {}
    for epsilon in EPSILONS:
        for name, attack, kwargs in (
            ('FGSM', fgsm_attack, {'epsilon': epsilon}),
            ('PGD', pgd_attack, {'epsilon': epsilon, 'alpha': 0.005, 'iters': 20, 'restarts': 2}),
        ):
            cached = load_or_generate_adversarial_dataset(
                model, test_loader, criterion, name.lower(), attack, kwargs
            )
            if epsilon == PREVIEW_EPSILON:
                # Test loading is sequential, so cache row zero matches test dataset item zero.
                perturbed_samples[name] = cached['images'][:1].clone()
            adv_accuracy, asr = score_perturbed_images(model, cached)
            if epsilon == 0 and abs(adv_accuracy - accuracy) > 1e-12:
                raise ValueError('Epsilon-zero accuracy must match clean accuracy')
            rows.append({'attack': name, **kwargs, 'adversarial_accuracy': adv_accuracy,
                         'attack_success_rate': asr, 'clean_accuracy': accuracy,
                         'accuracy_change': adv_accuracy - accuracy})
            print(f'{name} epsilon={epsilon}: accuracy={adv_accuracy:.4f}, ASR={asr}')
            for transform_name, transform, parameters in TRANSFORMATIONS:
                transformed_accuracy, transformed_asr = score_perturbed_images(
                    model, cached, transform, parameters
                )
                transformation_rows.append({
                    'attack': name, **kwargs, 'transformation': transform_name,
                    'transformation_parameters': parameters,
                    'transformed_accuracy': transformed_accuracy,
                    'attack_success_rate': transformed_asr,
                    'untransformed_adversarial_accuracy': adv_accuracy,
                    'untransformed_attack_success_rate': asr,
                    'accuracy_change': transformed_accuracy - adv_accuracy,
                })
                print(f'  {transform_name}: accuracy={transformed_accuracy:.4f}, ASR={transformed_asr}')
    (ROOT / 'baseline_results.json').write_text(json.dumps(
        {'seed': SEED, 'model_hash': model_fingerprint(model), 'preprocessing': PREPROCESSING,
         'clean_macro_f1': macro_f1, 'results': rows}, indent=2), encoding='utf-8')

    (ROOT / 'transformation_results.json').write_text(json.dumps(
        {'seed': SEED, 'model_hash': model_fingerprint(model), 'preprocessing': PREPROCESSING,
         'clean_accuracy': accuracy, 'results': transformation_rows}, indent=2), encoding='utf-8')

    if perturbed_samples:
        clean_image, label = test_loader.dataset[0]
        show_image_comparison(clean_image, label, perturbed_samples, PREVIEW_EPSILON)


if __name__ == '__main__':
    main()


100%|██████████| 187M/187M [00:13<00:00, 13.7MB/s] 
100%|██████████| 89.0M/89.0M [00:10<00:00, 8.78MB/s]
100%|██████████| 99.6k/99.6k [00:00<00:00, 211kB/s]


Epoch 1/10: train loss=3.0053, validation loss=1.8548, validation accuracy=0.4523
Epoch 2/10: train loss=1.1546, validation loss=0.4537, validation accuracy=0.8544
Epoch 3/10: train loss=0.3592, validation loss=0.1517, validation accuracy=0.9523
Epoch 4/10: train loss=0.1709, validation loss=0.0778, validation accuracy=0.9803
Epoch 5/10: train loss=0.0995, validation loss=0.0590, validation accuracy=0.9850
Epoch 6/10: train loss=0.0764, validation loss=0.0583, validation accuracy=0.9839
Epoch 7/10: train loss=0.0563, validation loss=0.0524, validation accuracy=0.9861
Epoch 8/10: train loss=0.0532, validation loss=0.0375, validation accuracy=0.9901
Epoch 9/10: train loss=0.0401, validation loss=0.0339, validation accuracy=0.9910
Epoch 10/10: train loss=0.0337, validation loss=0.0194, validation accuracy=0.9947
Restored best validation checkpoint from epoch 10
Clean test loss=0.2835, accuracy=0.9465, macro F1=0.9102
Saved FGSM images: v2_fgsm_f00a7b5f80ea06920c4dedfb24728fbed81ccf17461c8